In [ ]:
# # =====================================================
# # IMPORTS
# # =====================================================
# import numpy as np
# import pandas as pd
# import os

# from sklearn.model_selection import train_test_split, GridSearchCV
# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# from sklearn.metrics import (
#     accuracy_score,
#     classification_report,
#     roc_auc_score,
#     silhouette_score
# )

# from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
# from sklearn.neighbors import NearestCentroid

# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import (
#     RandomForestClassifier,
#     StackingClassifier,
#     AdaBoostClassifier
# )
# from sklearn.naive_bayes import GaussianNB
# from sklearn.tree import DecisionTreeClassifier

# from xgboost import XGBClassifier
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline


# # =====================================================
# # 1. CARGAR DATOS
# # =====================================================
# def cargar_y_preparar_datos(ruta_archivo):
#     df = pd.read_excel(ruta_archivo)

#     df_viv = df[df['Proposito'].astype(str)
#                 .str.contains('Vivienda', case=False, na=False)].copy()

#     df_viv['Impago_Label'] = df_viv['Impago'].map({0: 0, 1: 1})

#     return df_viv


# ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')
# df = cargar_y_preparar_datos(ruta_real)

# df.columns

# target_col = "Impago_Label"
# columnas_a_eliminar = [
#     "ID",
#     "Impago",
#     "Prima",
#     "Proposito"
# ]

# y = df[target_col]
# X = df.drop(columns=[target_col])
# X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

# if "Impago" in X.columns:
#     X = X.drop(columns=["Impago"])

# for col in X.columns:
#     if "id" in col.lower():
#         X = X.drop(columns=[col])

# high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
# X = X.drop(columns=high_card_cols)

# cat_cols = X.select_dtypes(include="object").columns
# X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# X = X.astype("float32")


# # =====================================================
# # 2. TRAIN / TEST SPLIT
# # =====================================================
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=0.25,
#     stratify=y,
#     random_state=42
# )


# # =====================================================
# # 3. CLUSTERING CORRECTO (SIN LEAKAGE)
# # =====================================================
# scaler_cluster = StandardScaler()
# X_train_cluster = scaler_cluster.fit_transform(X_train)
# X_test_cluster = scaler_cluster.transform(X_test)


# def asignar_clusters(model, X_train_scaled, X_test_scaled):
#     labels_train = model.fit_predict(X_train_scaled)

#     # Para modelos sin predict usamos NearestCentroid
#     if hasattr(model, "predict"):
#         labels_test = model.predict(X_test_scaled)
#     else:
#         centroid_clf = NearestCentroid()
#         centroid_clf.fit(X_train_scaled, labels_train)
#         labels_test = centroid_clf.predict(X_test_scaled)

#     return labels_train, labels_test


# # Evaluar los tres
# scores = {}

# # KMeans
# kmeans = KMeans(n_clusters=3, random_state=42)
# labels_k_train, labels_k_test = asignar_clusters(
#     kmeans, X_train_cluster, X_test_cluster
# )
# scores["KMeans"] = silhouette_score(X_train_cluster, labels_k_train)

# # Agglomerative
# agg = AgglomerativeClustering(n_clusters=3)
# labels_a_train, labels_a_test = asignar_clusters(
#     agg, X_train_cluster, X_test_cluster
# )
# scores["Agglomerative"] = silhouette_score(X_train_cluster, labels_a_train)

# # DBSCAN
# db = DBSCAN(eps=2, min_samples=10)
# labels_d_train, labels_d_test = asignar_clusters(
#     db, X_train_cluster, X_test_cluster
# )

# if len(set(labels_d_train)) > 1:
#     scores["DBSCAN"] = silhouette_score(X_train_cluster, labels_d_train)
# else:
#     scores["DBSCAN"] = -1


# best_cluster = max(scores, key=scores.get)
# print("Mejor clustering:", best_cluster)

# X_train = X_train.copy()
# X_test = X_test.copy()

# if best_cluster == "KMeans":
#     X_train["cluster_feature"] = labels_k_train
#     X_test["cluster_feature"] = labels_k_test
# elif best_cluster == "Agglomerative":
#     X_train["cluster_feature"] = labels_a_train
#     X_test["cluster_feature"] = labels_a_test
# else:
#     X_train["cluster_feature"] = labels_d_train
#     X_test["cluster_feature"] = labels_d_test


# =====================================================
# 4. FUNCIÓN ENTRENAMIENTO
# =====================================================
# def entrenar_modelo(
#     nombre_modelo,
#     modelo,
#     param_grid,
#     X_train, X_test,
#     y_train, y_test,
#     usar_smote=False,
#     usar_pca=False
# ):

#     steps = []

#     steps.append(("scaler", StandardScaler()))

#     if usar_smote:
#         steps.append(("smote", SMOTE(random_state=42)))

#     if usar_pca:
#         steps.append(("pca", PCA(n_components=0.95, random_state=42)))

#     steps.append(("model", modelo))

#     pipe = ImbPipeline(steps)

#     param_grid_pipeline = {
#         f"model__{k}": v for k, v in param_grid.items()
#     }

#     grid = GridSearchCV(
#         pipe,
#         param_grid_pipeline,
#         cv=3,
#         scoring="roc_auc",
#         n_jobs=-1
#     )

#     grid.fit(X_train, y_train)

#     y_pred = grid.best_estimator_.predict(X_test)
#     y_proba = grid.best_estimator_.predict_proba(X_test)[:, 1]

#     acc = accuracy_score(y_test, y_pred)
#     roc = roc_auc_score(y_test, y_proba)

#     print("\n", "="*60)
#     print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca}")
#     print("Best params:", grid.best_params_)
#     print("Accuracy:", round(acc, 4))
#     print("ROC-AUC:", round(roc, 4))
#     print(classification_report(y_test, y_pred))

#     return {
#         "Modelo": nombre_modelo,
#         "SMOTE": usar_smote,
#         "PCA": usar_pca,
#         "Accuracy": acc,
#         "ROC_AUC": roc
#     }


# # =====================================================
# # 5. MODELOS
# # =====================================================
# modelos = {

#     "LogReg": (
#         LogisticRegression(max_iter=1000),
#         {"C": [0.01, 0.1, 1]}
#     ),

#     "RandomForest": (
#         RandomForestClassifier(random_state=42),
#         {"n_estimators": [100, 200]}
#     ),

#     "DecisionTree": (
#         DecisionTreeClassifier(random_state=42),
#         {"max_depth": [None, 5, 10]}
#     ),

#     "AdaBoost": (
#         AdaBoostClassifier(random_state=42),
#         {"n_estimators": [50, 100]}
#     ),

#     "XGBoost": (
#         XGBClassifier(
#             eval_metric="logloss",
#             random_state=42,
#             use_label_encoder=False
#         ),
#         {
#             "n_estimators": [100],
#             "max_depth": [3, 6]
#         }
#     )
# }

# estimadores_base = [
#     ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
#     ("dt", DecisionTreeClassifier(random_state=42)),
#     ("nb", GaussianNB())
# ]

# stacking = StackingClassifier(
#     estimators=estimadores_base,
#     final_estimator=LogisticRegression()
# )

# modelos["Stacking"] = (
#     stacking,
#     {"final_estimator__C": [0.1, 1]}
# )


# # =====================================================
# # 6. EJECUCIÓN
# # =====================================================
# combinaciones = [
#     (False, False),
#     (True, False),
#     (False, True),
#     (True, True)
# ]

# resultados_finales = []

# for nombre, (modelo, grid) in modelos.items():
#     for smote_flag, pca_flag in combinaciones:

#         res = entrenar_modelo(
#             nombre,
#             modelo,
#             grid,
#             X_train, X_test,
#             y_train, y_test,
#             smote_flag,
#             pca_flag
#         )

#         resultados_finales.append(res)

# df_resultados = pd.DataFrame(resultados_finales)

# print("\n=========== RESULTADOS FINALES ===========")
# print(df_resultados.sort_values("ROC_AUC", ascending=False))

Mejor clustering: DBSCAN

LogReg | SMOTE=False | PCA=False
Best params: {'model__C': 0.01}
Accuracy: 0.8858
ROC-AUC: 0.6131
              precision    recall  f1-score   support

           0       0.89      1.00      0.94      2419
           1       0.00      0.00      0.00       312

    accuracy                           0.89      2731
   macro avg       0.44      0.50      0.47      2731
weighted avg       0.78      0.89      0.83      2731



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


LogReg | SMOTE=True | PCA=False
Best params: {'model__C': 0.01}
Accuracy: 0.5591
ROC-AUC: 0.6127
              precision    recall  f1-score   support

           0       0.91      0.55      0.69      2419
           1       0.15      0.60      0.24       312

    accuracy                           0.56      2731
   macro avg       0.53      0.58      0.46      2731
weighted avg       0.83      0.56      0.64      2731


LogReg | SMOTE=False | PCA=True
Best params: {'model__C': 0.01}
Accuracy: 0.8858
ROC-AUC: 0.6085
              precision    recall  f1-score   support

           0       0.89      1.00      0.94      2419
           1       0.00      0.00      0.00       312

    accuracy                           0.89      2731
   macro avg       0.44      0.50      0.47      2731
weighted avg       0.78      0.89      0.83      2731



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


LogReg | SMOTE=True | PCA=True
Best params: {'model__C': 0.01}
Accuracy: 0.5617
ROC-AUC: 0.6078
              precision    recall  f1-score   support

           0       0.91      0.56      0.69      2419
           1       0.15      0.59      0.24       312

    accuracy                           0.56      2731
   macro avg       0.53      0.58      0.46      2731
weighted avg       0.83      0.56      0.64      2731


RandomForest | SMOTE=False | PCA=False
Best params: {'model__n_estimators': 100}
Accuracy: 0.8502
ROC-AUC: 0.519
              precision    recall  f1-score   support

           0       0.89      0.95      0.92      2419
           1       0.11      0.04      0.06       312

    accuracy                           0.85      2731
   macro avg       0.50      0.50      0.49      2731
weighted avg       0.80      0.85      0.82      2731


RandomForest | SMOTE=True | PCA=False
Best params: {'model__n_estimators': 100}
Accuracy: 0.825
ROC-AUC: 0.5166
              precisio

...


XGBoost | SMOTE=True | PCA=True
Best params: {'model__max_depth': 3, 'model__n_estimators': 100}
Accuracy: 0.6514
ROC-AUC: 0.5706
              precision    recall  f1-score   support

           0       0.90      0.68      0.78      2419
           1       0.14      0.41      0.21       312

    accuracy                           0.65      2731
   macro avg       0.52      0.55      0.49      2731
weighted avg       0.81      0.65      0.71      2731


Stacking | SMOTE=False | PCA=False
Best params: {'model__final_estimator__C': 1}
Accuracy: 0.8858
ROC-AUC: 0.6001
              precision    recall  f1-score   support

           0       0.89      1.00      0.94      2419
           1       0.00      0.00      0.00       312

    accuracy                           0.89      2731
   macro avg       0.44      0.50      0.47      2731
weighted avg       0.78      0.89      0.83      2731



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


Stacking | SMOTE=True | PCA=False
Best params: {'model__final_estimator__C': 0.1}
Accuracy: 0.8151
ROC-AUC: 0.5314
              precision    recall  f1-score   support

           0       0.89      0.91      0.90      2419
           1       0.12      0.10      0.11       312

    accuracy                           0.82      2731
   macro avg       0.51      0.50      0.50      2731
weighted avg       0.80      0.82      0.81      2731


Stacking | SMOTE=False | PCA=True
Best params: {'model__final_estimator__C': 1}
Accuracy: 0.8858
ROC-AUC: 0.5954
              precision    recall  f1-score   support

           0       0.89      1.00      0.94      2419
           1       0.00      0.00      0.00       312

    accuracy                           0.89      2731
   macro avg       0.44      0.50      0.47      2731
weighted avg       0.78      0.89      0.83      2731



c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\unaip\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


Stacking | SMOTE=True | PCA=True
Best params: {'model__final_estimator__C': 0.1}
Accuracy: 0.7759
ROC-AUC: 0.5336
              precision    recall  f1-score   support

           0       0.89      0.86      0.87      2419
           1       0.11      0.14      0.12       312

    accuracy                           0.78      2731
   macro avg       0.50      0.50      0.50      2731
weighted avg       0.80      0.78      0.79      2731


=========== RESULTADOS FINALES ===========
          Modelo  SMOTE    PCA  Accuracy   ROC_AUC
0         LogReg  False  False  0.885756  0.613110
1         LogReg   True  False  0.559136  0.612737
14      AdaBoost  False   True  0.885756  0.611034
2         LogReg  False   True  0.885756  0.608531
3         LogReg   True   True  0.561699  0.607838
12      AdaBoost  False  False  0.885756  0.606968
13      AdaBoost   True  False  0.849140  0.602442
20      Stacking  False  False  0.885756  0.600058
15      AdaBoost   True   True  0.560234  0.598814
22  

Precision, Recall, F1-score
Clase 1 = impago

Precision: De los clientes que el modelo predice como impago, cuántos realmente incumplen.

Recall: De todos los clientes que realmente incumplen, cuántos el modelo predice correctamente.

F1-score: Media armónica entre precision y recall, útil para balancear ambos.


Tu objetivo es detectar impagos (clase 1). Por tanto:

Recall de clase 1 es la métrica más importante
→ quieres minimizar falsos negativos (clientes que incumplen pero tu modelo dice que no).

Precision importa menos que recall si estás dispuesto a aceptar algunos falsos positivos (alertas de riesgo innecesarias).

F1-score de clase 1 te da un balance, útil para comparar modelos.

ROC-AUC es buena métrica general de ranking de riesgo.

In [ ]:
# =====================================================
# IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    make_scorer,
    silhouette_score,
    precision_recall_curve
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# =====================================================
# 1. CARGAR DATOS
# =====================================================
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')
df = cargar_y_preparar_datos(ruta_real)

# =====================================================
# 2. DEFINIR X e y
# =====================================================
target_col = "Impago_Label"
columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

y = df[target_col]
X = df.drop(columns=[target_col])
X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

# Eliminar alta cardinalidad
high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
X = X.drop(columns=high_card_cols)

# One-hot encoding
cat_cols = X.select_dtypes(include="object").columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X = X.astype("float32")

# =====================================================
# 3. TRAIN / TEST SPLIT
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

# =====================================================
# 4. CLUSTERING SIN LEAKAGE
# =====================================================
# =====================================================
# 4. CLUSTERING: TORNEO K-MEANS vs AGLOMERATIVO
# =====================================================
from sklearn.neighbors import NearestCentroid # Necesario para predecir con Aglomerativo

print("--- Iniciando Torneo de Clustering (Feature Engineering) ---")

# 1. Escalado (Solo para calcular distancias, evitamos data leakage)
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

# Diccionario para guardar resultados temporales
scores = {}
labels_storage = {}

# --- FUNCIÓN AUXILIAR PARA PREDECIR EN TEST ---
def asignar_clusters(model, X_train_scaled, X_test_scaled):
    labels_train = model.fit_predict(X_train_scaled)
    
    if hasattr(model, "predict"):
        # K-Means tiene predict nativo
        labels_test = model.predict(X_test_scaled)
    else:
        # Aglomerativo NO tiene predict, usamos NearestCentroid
        centroid_clf = NearestCentroid()
        centroid_clf.fit(X_train_scaled, labels_train)
        labels_test = centroid_clf.predict(X_test_scaled)
        
    return labels_train, labels_test

# --- MODELO A: K-MEANS ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
k_labels_train, k_labels_test = asignar_clusters(kmeans, X_train_cluster, X_test_cluster)

score_k = silhouette_score(X_train_cluster, k_labels_train)
scores["KMeans"] = score_k
labels_storage["KMeans"] = (k_labels_train, k_labels_test)
print(f"Silhouette KMeans: {score_k:.4f}")

# --- MODELO B: AGLOMERATIVO (JERÁRQUICO) ---
agg = AgglomerativeClustering(n_clusters=3)
a_labels_train, a_labels_test = asignar_clusters(agg, X_train_cluster, X_test_cluster)

score_a = silhouette_score(X_train_cluster, a_labels_train)
scores["Agglomerative"] = score_a
labels_storage["Agglomerative"] = (a_labels_train, a_labels_test)
print(f"Silhouette Agglomerative: {score_a:.4f}")

# --- SELECCIÓN DEL GANADOR ---
best_cluster_name = max(scores, key=scores.get)
print(f"\n🏆 GANADOR: {best_cluster_name}")

# Recuperamos las etiquetas del modelo ganador
final_labels_train, final_labels_test = labels_storage[best_cluster_name]

# --- APLICAMOS ONE-HOT ENCODING (SOLO DEL GANADOR) ---
# Convertimos el cluster 0, 1, 2 en columnas dummies (Cluster_Group_0, Cluster_Group_1...)
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Aseguramos que test tenga las mismas columnas que train (rellenando con 0 si falta algún cluster)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Asignamos el índice original para que coincida con X_train/y_train al concatenar
train_dummies.index = X_train.index
test_dummies.index = X_test.index

# Concatenamos las nuevas columnas al dataset original
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f"✅ Variables de cluster añadidas al dataset usando {best_cluster_name}")
print(f"Nuevas columnas generadas: {list(train_dummies.columns)}")
# =====================================================
# 5. FUNCIÓN ENTRENAMIENTO PRIORITIZANDO RECALL
# =====================================================
def entrenar_modelo(
    nombre_modelo,
    modelo,
    param_grid,
    X_train, X_test,
    y_train, y_test,
    usar_smote=False,
    usar_pca=False,
    threshold=0.3
):

    steps = [("scaler", StandardScaler())]

    if usar_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    if usar_pca:
        steps.append(("pca", PCA(n_components=0.95, random_state=42)))

    steps.append(("model", modelo))
    pipe = ImbPipeline(steps)

    param_grid_pipeline = {f"model__{k}": v for k,v in param_grid.items()}

    recall_scorer = make_scorer(recall_score, pos_label=1)

    grid = GridSearchCV(pipe, param_grid_pipeline, cv=3, scoring=recall_scorer, n_jobs=-1)
    grid.fit(X_train, y_train)

    # Predicciones ajustando threshold
    y_proba = grid.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    recall1 = recall_score(y_test, y_pred, pos_label=1)

    print("\n", "="*60)
    print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={threshold}")
    print("Best params:", grid.best_params_)
    print("Accuracy:", round(acc,4))
    print("ROC-AUC:", round(roc,4))
    print("Recall clase 1:", round(recall1,4))
    print(classification_report(y_test, y_pred))

    return {
        "Modelo": nombre_modelo,
        "SMOTE": usar_smote,
        "PCA": usar_pca,
        "Threshold": threshold,
        "Accuracy": acc,
        "ROC_AUC": roc,
        "Recall_1": recall1
    }

# =====================================================
# 6. DEFINIR MODELOS
# =====================================================
modelos = {
    "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C":[0.01,0.1,1]}),
    "RandomForest": (RandomForestClassifier(random_state=42, class_weight="balanced"), {"n_estimators":[100,200]}),
    "DecisionTree": (DecisionTreeClassifier(random_state=42, class_weight="balanced"), {"max_depth":[None,5,10]}),
    "AdaBoost": (AdaBoostClassifier(random_state=42), {"n_estimators":[50,100]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
                {"n_estimators":[100], "max_depth":[3,6]})
}

estimadores_base = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("nb", GaussianNB())
]

stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression()
)

modelos["Stacking"] = (stacking, {"final_estimator__C":[0.1,1]})

# =====================================================
# 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
# =====================================================
combinaciones = [
    (False, False),
    (True, False),
    (False, True),
    (True, True)
]

threshold = 0.4 

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:
        res = entrenar_modelo(
            nombre, modelo, grid,
            X_train, X_test,
            y_train, y_test,
            usar_smote=smote_flag,
            usar_pca=pca_flag,
            threshold=threshold
        )
        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

print("\n=========== RESULTADOS FINALES ===========")
print(df_resultados.sort_values("Recall_1", ascending=False))

ModuleNotFoundError: No module named 'imblearn'

In [10]:
print("\n=========== RESULTADOS FINALES ===========")
print(df_resultados.sort_values("Recall_1", ascending=False))


=========== RESULTADOS FINALES ===========
          Modelo  SMOTE    PCA  Threshold  Accuracy   ROC_AUC  Recall_1
13      AdaBoost   True  False        0.4  0.121201  0.594209  0.996795
15      AdaBoost   True   True        0.4  0.207616  0.598814  0.935897
0         LogReg  False  False        0.4  0.355547  0.613183  0.862179
2         LogReg  False   True        0.4  0.358843  0.608082  0.852564
3         LogReg   True   True        0.4  0.390333  0.607797  0.839744
1         LogReg   True  False        0.4  0.389235  0.612998  0.836538
8   DecisionTree  False  False        0.4  0.369096  0.585460  0.814103
10  DecisionTree  False   True        0.4  0.367997  0.557088  0.766026
11  DecisionTree   True   True        0.4  0.344196  0.564939  0.762821
9   DecisionTree   True  False        0.4  0.472354  0.587178  0.737179
19       XGBoost   True   True        0.4  0.531673  0.570622  0.602564
4   RandomForest  False  False        0.4  0.741487  0.525289  0.221154
6   RandomForest  Fa

In [3]:
# =====================================================
# IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    make_scorer,
    silhouette_score,
    precision_recall_curve
)

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# =====================================================
# 1. CARGAR DATOS
# =====================================================
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')
df = cargar_y_preparar_datos(ruta_real)

# =====================================================
# 2. DEFINIR X e y
# =====================================================
target_col = "Impago_Label"
columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

y = df[target_col]
X = df.drop(columns=[target_col])
X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

# Eliminar alta cardinalidad
high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
X = X.drop(columns=high_card_cols)

# One-hot encoding
cat_cols = X.select_dtypes(include="object").columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X = X.astype("float32")

# =====================================================
# 3. TRAIN / TEST SPLIT
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

# =====================================================
# 4. CLUSTERING SIN LEAKAGE
# =====================================================
# =====================================================
# 4. CLUSTERING: TORNEO K-MEANS vs AGLOMERATIVO
# =====================================================
from sklearn.neighbors import NearestCentroid # Necesario para predecir con Aglomerativo

print("--- Iniciando Torneo de Clustering (Feature Engineering) ---")

# 1. Escalado (Solo para calcular distancias, evitamos data leakage)
scaler_cluster = StandardScaler()
X_train_cluster = scaler_cluster.fit_transform(X_train)
X_test_cluster = scaler_cluster.transform(X_test)

# Diccionario para guardar resultados temporales
scores = {}
labels_storage = {}

# --- FUNCIÓN AUXILIAR PARA PREDECIR EN TEST ---
def asignar_clusters(model, X_train_scaled, X_test_scaled):
    labels_train = model.fit_predict(X_train_scaled)
    
    if hasattr(model, "predict"):
        # K-Means tiene predict nativo
        labels_test = model.predict(X_test_scaled)
    else:
        # Aglomerativo NO tiene predict, usamos NearestCentroid
        centroid_clf = NearestCentroid()
        centroid_clf.fit(X_train_scaled, labels_train)
        labels_test = centroid_clf.predict(X_test_scaled)
        
    return labels_train, labels_test

# --- MODELO A: K-MEANS ---
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
k_labels_train, k_labels_test = asignar_clusters(kmeans, X_train_cluster, X_test_cluster)

score_k = silhouette_score(X_train_cluster, k_labels_train)
scores["KMeans"] = score_k
labels_storage["KMeans"] = (k_labels_train, k_labels_test)
print(f"Silhouette KMeans: {score_k:.4f}")

# --- MODELO B: AGLOMERATIVO (JERÁRQUICO) ---
agg = AgglomerativeClustering(n_clusters=3)
a_labels_train, a_labels_test = asignar_clusters(agg, X_train_cluster, X_test_cluster)

score_a = silhouette_score(X_train_cluster, a_labels_train)
scores["Agglomerative"] = score_a
labels_storage["Agglomerative"] = (a_labels_train, a_labels_test)
print(f"Silhouette Agglomerative: {score_a:.4f}")

# --- SELECCIÓN DEL GANADOR ---
best_cluster_name = max(scores, key=scores.get)
print(f"\n🏆 GANADOR: {best_cluster_name}")

# Recuperamos las etiquetas del modelo ganador
final_labels_train, final_labels_test = labels_storage[best_cluster_name]

# --- APLICAMOS ONE-HOT ENCODING (SOLO DEL GANADOR) ---
# Convertimos el cluster 0, 1, 2 en columnas dummies (Cluster_Group_0, Cluster_Group_1...)
train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

# Aseguramos que test tenga las mismas columnas que train (rellenando con 0 si falta algún cluster)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

# Asignamos el índice original para que coincida con X_train/y_train al concatenar
train_dummies.index = X_train.index
test_dummies.index = X_test.index

# Concatenamos las nuevas columnas al dataset original
X_train = pd.concat([X_train, train_dummies], axis=1)
X_test = pd.concat([X_test, test_dummies], axis=1)

print(f"✅ Variables de cluster añadidas al dataset usando {best_cluster_name}")
print(f"Nuevas columnas generadas: {list(train_dummies.columns)}")
# =====================================================
# 5. FUNCIÓN ENTRENAMIENTO PRIORITIZANDO RECALL
# =====================================================
def entrenar_modelo(
    nombre_modelo,
    modelo,
    param_grid,
    X_train, X_test,
    y_train, y_test,
    usar_smote=False,
    usar_pca=False,
    threshold=None # Si es None, lo calcula dinámicamente
):

    steps = [("scaler", StandardScaler())]

    if usar_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    if usar_pca:
        steps.append(("pca", PCA(n_components=0.95, random_state=42)))

    steps.append(("model", modelo))
    pipe = ImbPipeline(steps)

    param_grid_pipeline = {f"model__{k}": v for k,v in param_grid.items()}

    recall_scorer = make_scorer(recall_score, pos_label=1)

    grid = GridSearchCV(pipe, param_grid_pipeline, cv=3, scoring=recall_scorer, n_jobs=-1)
    grid.fit(X_train, y_train)

    y_proba = grid.best_estimator_.predict_proba(X_test)[:, 1]

    if threshold is None:  #treshold dinamico para basándose en los datos
        precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        
        print(f"   >>> Umbral Óptimo calculado (Max F1): {best_threshold:.4f}")
    else:
        best_threshold = threshold

    y_pred = (y_proba >= best_threshold).astype(int)

    # Métricas
    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    recall1 = recall_score(y_test, y_pred, pos_label=1)
    precision1 = precision_score(y_test, y_pred, pos_label=1, zero_division=0)

    print("\n", "="*60)
    print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={best_threshold:.4f}")
    print("Best params:", grid.best_params_)
    print("ROC-AUC:", round(roc,4))
    print("Recall (Impago):", round(recall1,4))
    print("Precision (Impago):", round(precision1,4))
    print(classification_report(y_test, y_pred))

    return {
        "Modelo": nombre_modelo,
        "SMOTE": usar_smote,
        "PCA": usar_pca,
        "Threshold": best_threshold,
        "Accuracy": acc,
        "ROC_AUC": roc,
        "Recall_1": recall1,
        "Precision_1": precision1
    }

# =====================================================
# 6. DEFINIR MODELOS
# =====================================================
modelos = {
    "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C":[0.01,0.1,1]}),
    "RandomForest": (RandomForestClassifier(random_state=42, class_weight="balanced"), {"n_estimators":[100,200]}),
    "DecisionTree": (DecisionTreeClassifier(random_state=42, class_weight="balanced"), {"max_depth":[None,5,10]}),
    "AdaBoost": (AdaBoostClassifier(random_state=42), {"n_estimators":[50,100]}),
    "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
                {"n_estimators":[100], "max_depth":[3,6]})
}

estimadores_base = [
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("nb", GaussianNB())
]

stacking = StackingClassifier(
    estimators=estimadores_base,
    final_estimator=LogisticRegression()
)

modelos["Stacking"] = (stacking, {"final_estimator__C":[0.1,1]})

# =====================================================
# 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
# =====================================================
combinaciones = [
    (False, False),
    (True, False),
]

resultados_finales = []

for nombre, (modelo, grid) in modelos.items():
    for smote_flag, pca_flag in combinaciones:
        
        res = entrenar_modelo(
            nombre, modelo, grid,
            X_train, X_test,
            y_train, y_test,
            usar_smote=smote_flag,
            usar_pca=pca_flag,
            threshold=None  # <--- ¡IMPORTANTE! Poner None aquí
        )
        
        resultados_finales.append(res)

df_resultados = pd.DataFrame(resultados_finales)

print("\n=========== RESULTADOS FINALES ===========")
# Ordenamos por Recall porque es tu prioridad en Riesgos
print(df_resultados.sort_values("Recall_1", ascending=False))

ModuleNotFoundError: No module named 'imblearn'